<a href="https://colab.research.google.com/github/jackevansadl/chem2002/blob/main/C1/03-Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 3: Analysis and Thermodynamic Properties 📈

Welcome to the final notebook\! In the last section, you ran a successful simulation of liquid Argon and confirmed that it reached thermal equilibrium. Now, we'll analyze the data you generated to calculate a real, macroscopic thermodynamic property: the **constant volume heat capacity ($C\_v$)**.

This process is a perfect example of how computational chemistry acts as a bridge between the microscopic world of atomic motion and the macroscopic world of thermodynamics that you're learning about in your lectures.

First, let's import the libraries we'll need for analysis.

> ⏱️ **Time guide:** about **45 minutes**. Answer the 📋 and 💭 prompts in your workbook.

### What you will do
1. Load the data your simulation produced.
2. Find the **equilibration time** from the energy plot.
3. Turn the tiny **energy fluctuations** into a real thermodynamic property — the heat capacity $C_v$.

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from ase import units

-----
## 📤 Upload your data from Part 2 — IMPORTANT

This notebook needs the **`md.log`** and **`md.traj`** files you produced (and downloaded) at the end of **Part 2**. Because Colab runs each notebook in a fresh session, you must upload them here first.

Run the cell below and select **both** `md.log` and `md.traj` from your computer. *(If you are not on Colab and the files are already in this folder, you can skip this cell.)*

In [ ]:
# Upload the md.log and md.traj files you downloaded at the end of Part 2.
try:
    from google.colab import files
    uploaded = files.upload()   # select BOTH md.log and md.traj
except ModuleNotFoundError:
    print("Not running on Colab — make sure md.log and md.traj are in this folder.")

-----

## 3.1 Loading the Simulation Data

The `MDLogger` we used in the previous notebook saved all the key properties of our system at regular intervals into the file `md.log`. Let's load that data into our notebook using NumPy.

In [ ]:
# Load the data from the log file, skipping the header row
log_data = np.loadtxt('md.log', skiprows=1)

# Let's assign columns to named variables for clarity
# Column indices: 0=Time(ps), 4=Temp(K), 1=E_tot(eV)
time = log_data[:, 0]
temperature = log_data[:, 4]
total_energy = log_data[:, 1]

print("Log data loaded successfully!")

-----

## 3.2 Visualizing System Energy & Determining Equilibration Time

In the last notebook, we plotted temperature to check for equilibration. Plotting the **total energy** is another excellent way to do this. We're looking for the point where the energy stops decreasing and starts fluctuating around a stable average. This stable region is the "production" phase, and it's the only part of the data we should use for calculating properties.


In [ ]:
# Plotting the Total Energy
plt.figure(figsize=(8, 5))
plt.plot(time, total_energy, 'o-')
plt.title('Total System Energy During Simulation')
plt.xlabel('Time [ps]')
plt.ylabel('Total Energy [eV]')
plt.grid(True)
plt.show()

📖 **Reading the plot.** Early in the run the total energy drifts as the system settles from its artificial starting point — this is **equilibration**. Once it stops drifting and just **wobbles around a steady average**, the system is at equilibrium; this later part is the **production** region. We keep only the production data, otherwise the equilibration drift biases our averages.

### **📋 Reporting Task 6**

Based on the **Total Energy vs. Time** plot above:

1.  Save a copy of the plot for your workbook.
2.  Estimate the time (in picoseconds) at which the system appears to be fully equilibrated. We will use this value to slice our data for the final calculation.

In [ ]:
# Answer goes here

-----

## 3.3 Calculating Heat Capacity from Fluctuations

### The Theory

In statistical mechanics, there's a beautiful connection between macroscopic properties and the microscopic fluctuations of a system at equilibrium. The constant volume heat capacity ($C\_v$) can be calculated from the **variance** (the square of the standard deviation, $\\sigma\_E^2$) of the total energy ($E$) using the following formula:

$$C_v = \frac{\langle E^2 \rangle - \langle E \rangle^2}{k_B T^2} = \frac{\sigma_E^2}{k_B T^2}$$

Here, $k\_B$ is the Boltzmann constant and $T$ is the average temperature. We have all of this information in our log file\!

### The Calculation

Now, let's write the code to perform this calculation. We'll start our analysis after the equilibration time you determined previously (a value around 2 ps is a good choice).


📖 **Why do fluctuations give a heat capacity?** A system that can soak up energy *easily* (high heat capacity) shows **large** energy fluctuations at a given temperature; a "stiff" system shows small ones. The formula $C_v = \sigma_E^2 / (k_B T^2)$ makes this precise — it reads the heat capacity straight out of the **size of the wobble** in the energy. A remarkable bridge from a microscopic jiggle to a macroscopic, measurable property.

In [ ]:
# --- Calculation of Cv ---

# 1. Define the equilibration time (in ps) based on your plot
equilibration_time = 2.0  # ps

# Find the index where the production phase starts
start_index = np.where(time >= equilibration_time)[0][0]

# 2. Slice the arrays to use only the production (equilibrated) data
production_energy = total_energy[start_index:]
production_temperature = temperature[start_index:]

# 3. Calculate the average temperature and the variance of the total energy
avg_temp = np.mean(production_temperature)
energy_variance = np.var(production_energy)   # variance = std**2, units eV^2

# 4. Number of atoms actually simulated (read from the trajectory so this stays
#    correct whatever box size you used in notebook 02)
from ase.io import read
num_atoms = len(read('md.traj', index=0))

# 5. Heat capacity of the whole system from the canonical fluctuation formula
#    Cv = <E^2 - <E>^2> / (kB * T^2)   -> units of eV/K
cv_system_eV_per_K = energy_variance / (units.kB * avg_temp**2)

# 6. Convert to a molar heat capacity in J / (K mol)
#    Divide by the number of atoms, then scale to one mole and convert eV -> J.
#    units.mol = Avogadro's number; units.J = number of eV in one Joule.
cv_J_per_K_mol = cv_system_eV_per_K / num_atoms * units.mol / units.J

# The experimental value for liquid argon is ~25 J / (K mol)
print(f"Number of atoms:     {num_atoms}")
print(f"Average Temperature: {avg_temp:.2f} K")
print(f"Energy Variance:     {energy_variance:.6f} eV^2")
print(f"Calculated Cv:       {cv_J_per_K_mol:.2f} J / (K mol)")

-----
### 🔧 Hands-on Exercise 3.A — How sensitive is $C_v$ to the cut-off? ⏱️ ~10 min

Your $C_v$ depends on where you decide equilibration ends. Re-run the calculation for a few `equilibration_time` values and watch how much the answer moves. **Predict:** if you include the noisy equilibration region (a *too-small* cut-off), will $C_v$ come out too **high** or too **low**?

In [ ]:
for eq_time in [0.0, 1.0, 2.0, 4.0]:
    start = np.where(time >= eq_time)[0][0]
    var = np.var(total_energy[start:])
    avgT = np.mean(temperature[start:])
    cv = var / (units.kB * avgT**2) / num_atoms * units.mol / units.J
    print(f"equilibration cut-off = {eq_time:.1f} ps  ->  Cv = {cv:.2f} J/K/mol")

-----
### 💭 Reflect
1. Why must we **exclude** the equilibration part of the run before calculating $C_v$?
2. Your simulated $C_v$ differs from the experimental ~25 J K⁻¹ mol⁻¹. Give **two** physical reasons (consider the size of the system, the cut-off in the potential, or the classical treatment of the atoms).


-----

### **📋 Reporting Task 7**

For the final part of your workbook, please provide:

1.  Your final calculated value for the constant volume heat capacity ($C_v$) in J K⁻¹ mol⁻¹.
2.  A short paragraph comparing your result to the experimental value. Compare your simulations to that reported by Nichele et al. (2017) 10.1016/j.molliq.2017.03.120 and provide **two** potential reasons why your simulated value might be different.

# Answer goes here.
